# Wine Quality — team coursework

**Trương Minh Hoàng · MSSV 25410008**

**ARCHIVED_SOURCE — source review, not a fresh benchmark.** Bản học tập lịch sử được giữ để giải thích quá trình làm bài. Output cũ và thông tin môi trường riêng đã được bỏ; xem README/AUDIT của dự án trước khi chạy. Historical calculations and known limitations are documented separately.


**Bài toán:** Phân tích và dự đoán chất lượng rượu vang (Wine Quality Prediction)

**Dữ liệu bạn cung cấp:**
- `winequality-red.csv` (1,599 mẫu)
- `winequality-white.csv` (4,898 mẫu)

**Mục tiêu trong notebook này (đúng theo file yêu cầu):**
- EDA (thống kê mô tả, phân bố, tương quan, mất cân bằng lớp)
- Tiền xử lý (kiểm tra missing, xử lý outliers *tùy chọn*, chuẩn hóa)
- Chuyển `quality` → nhãn nhị phân:  
  - `0` = chất lượng thấp (`quality ≤ 5`)  
  - `1` = chất lượng cao (`quality ≥ 6`)
- Train/Test với nhiều tỉ lệ: **0.1, 0.2, 0.3**
- Huấn luyện & so sánh ít nhất 4 mô hình:
  1. Logistic Regression (baseline)
  2. Decision Tree
  3. LightGBM
  4. XGBoost
- Đánh giá: Accuracy, Precision, Recall, F1, Confusion Matrix, ROC-AUC, ROC Curve
- Feature importance (LightGBM, XGBoost)
- (Optional) Regression dự đoán điểm `quality`


In [ ]:


#!pip -q install -U lightgbm xgboost


## 1️⃣ Import thư viện

In [ ]:
import os
import numpy as np
import pandas as pd

import math
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBClassifier, XGBRegressor

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42


## 2️⃣ Load dữ liệu

In [ ]:
# ✅ Load dữ liệu từ Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:


import pandas as pd
import glob


RED_PATH   = "/content/drive/MyDrive/Colab Notebooks/Học máy/do an/data/winequality-red.csv"
WHITE_PATH = "/content/drive/MyDrive/Colab Notebooks/Học máy/do an/data/winequality-white.csv"


df_red   = pd.read_csv(RED_PATH, sep=";")
df_white = pd.read_csv(WHITE_PATH, sep=";")

df_red["type"] = "red"
df_white["type"] = "white"
df_all = pd.concat([df_red, df_white], ignore_index=True)

print("Red shape  :", df_red.shape)
print("White shape:", df_white.shape)
print("All shape  :", df_all.shape)
display(df_all.head())


## 3️⃣ EDA — khám phá dữ liệu

In [ ]:
def basic_eda(df: pd.DataFrame, title: str):
    print(f"\n===== {title} =====")
    display(df.describe(include='all'))
    print("\nMissing values per column:")
    display(df.isna().sum().sort_values(ascending=False))

    plt.figure(figsize=(5,3))
    df["quality"].value_counts().sort_index().plot(kind="bar")
    plt.title(f"Quality distribution — {title}")
    plt.xlabel("Quality"); plt.ylabel("Count")
    plt.show()

    plt.figure(figsize=(5,3))
    (df["quality"] >= 6).astype(int).value_counts().sort_index().plot(kind="bar")
    plt.title(f"Binary label distribution (0=≤5, 1=≥6) — {title}")
    plt.xlabel("Label"); plt.ylabel("Count")
    plt.show()

    # Correlation heatmap (chỉ số)
    plt.figure(figsize=(10,6))
    corr = df.drop(columns=["type"], errors="ignore").corr(numeric_only=True)
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title(f"Correlation heatmap — {title}")
    plt.show()

basic_eda(df_all, "Combined (Red + White)")


### Biểu đồ phân bố từng đặc trưng (11 features)

In [ ]:


def plot_feature_distributions(df, feature_cols, title_prefix=""):
    n = len(feature_cols)
    ncols = 3
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 3))
    axes = axes.flatten()

    for i, col in enumerate(feature_cols):
        ax = axes[i]
        ax.hist(df[col].dropna(), bins=30)
        ax.set_title(f"{title_prefix}{col}")
        ax.set_xlabel(col)
        ax.set_ylabel("Count")

    # tắt các ô trống
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

# Phân bố từng đặc trưng trên toàn bộ dữ liệu (Combined)
plot_feature_distributions(df_all, [c for c in df_all.columns if c not in ["quality","type"]], title_prefix="All - ")

# (Tuỳ chọn) Boxplot để nhìn nhanh outliers cho 11 đặc trưng
feature_cols_eda = [c for c in df_all.columns if c not in ["quality","type"]]
plt.figure(figsize=(12, 6))
pd.DataFrame(df_all[feature_cols_eda]).plot(kind="box", vert=False)
plt.title("Boxplot các đặc trưng (kiểm tra outliers)")
plt.tight_layout()
plt.show()


## 4️⃣ Tiền xử lý dữ liệu

In [ ]:
# 4.1 Tạo nhãn phân loại
def make_binary_label(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["label"] = (out["quality"] >= 6).astype(int)   # 0: ≤5, 1: ≥6
    return out

# 4.2 (Tùy chọn) Loại outliers bằng IQR
def remove_outliers_iqr(df: pd.DataFrame, feature_cols, k=1.5):
    out = df.copy()
    mask = np.ones(len(out), dtype=bool)
    for col in feature_cols:
        q1 = out[col].quantile(0.25)
        q3 = out[col].quantile(0.75)
        iqr = q3 - q1
        lo = q1 - k * iqr
        hi = q3 + k * iqr
        mask &= out[col].between(lo, hi)
    return out.loc[mask].reset_index(drop=True)

df_all_labeled = make_binary_label(df_all)

FEATURE_COLS = [c for c in df_all_labeled.columns if c not in ["quality", "label", "type"]]
TARGET_COL = "label"

print("Feature cols:", FEATURE_COLS)
print("Label counts:")
display(df_all_labeled[TARGET_COL].value_counts())


## 5️⃣ Huấn luyện mô hình & đánh giá (nhiều tỉ lệ train/test)

In [ ]:

QUICK_RUN = True

def build_models():
    # Nếu QUICK_RUN=True thì giảm số cây để chạy nhanh
    n_estimators = 250 if QUICK_RUN else 800

    models = {}

    # 1) Logistic Regression (cần scale)
    models["LogisticRegression"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))
    ])

    # 2) Decision Tree
    models["DecisionTree"] = DecisionTreeClassifier(
        max_depth=6,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    )

    # 3) LightGBM
    models["LightGBM"] = LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=0.05,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    )

    # 4) XGBoost
    models["XGBoost"] = XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1
    )

    return models

def eval_one_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    # Một số model/pipeline có predict_proba
    y_prob = None
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)
        y_prob = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan,
    }

    return metrics, y_pred, y_prob

def run_experiment(df: pd.DataFrame, dataset_name: str, test_sizes=(0.1, 0.2, 0.3), drop_outliers=False):
    df_work = df.copy()

    # Tùy chọn bỏ outliers
    if drop_outliers:
        before = len(df_work)
        df_work = remove_outliers_iqr(df_work, FEATURE_COLS, k=1.5)
        after = len(df_work)
        print(f"[{dataset_name}] Outlier removal: {before} → {after}")

    X = df_work[FEATURE_COLS]
    y = df_work[TARGET_COL]

    models = build_models()
    all_rows = []
    artifacts = {}  # lưu để vẽ ROC/Confusion cho cấu hình tốt nhất

    for ts in test_sizes:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=ts, random_state=RANDOM_STATE, stratify=y
        )

        for name, model in models.items():
            m, y_pred, y_prob = eval_one_model(model, X_train, X_test, y_train, y_test)
            row = {"dataset": dataset_name, "test_size": ts, "model": name, **m}
            all_rows.append(row)

            artifacts[(dataset_name, ts, name)] = (model, X_test, y_test, y_pred, y_prob)

    results = pd.DataFrame(all_rows)
    results = results.sort_values(["dataset", "test_size", "f1"], ascending=[True, True, False]).reset_index(drop=True)
    return results, artifacts

# So sánh Wine đỏ / trắng là phần mở rộng (optional). Bạn có thể tắt nếu muốn chạy nhanh.
RUN_COMPARE_RED_WHITE = True

results_all, artifacts_all = run_experiment(df_all_labeled, "combined", test_sizes=(0.1,0.2,0.3), drop_outliers=False)

if RUN_COMPARE_RED_WHITE:
    results_red, artifacts_red = run_experiment(make_binary_label(df_red), "red", test_sizes=(0.1,0.2,0.3), drop_outliers=False)
    results_white, artifacts_white = run_experiment(make_binary_label(df_white), "white", test_sizes=(0.1,0.2,0.3), drop_outliers=False)
    results = pd.concat([results_all, results_red, results_white], ignore_index=True)
else:
    artifacts_red = artifacts_white = {}
    results = results_all.copy()

display(results.head(20))

# Bảng tổng hợp: trung bình theo dataset + model (gộp các test_size)
summary = (
    results.groupby(["dataset", "model"], as_index=False)[["accuracy","precision","recall","f1","roc_auc"]]
    .mean()
    .sort_values(["dataset","f1"], ascending=[True, False])
)
print("\n=== Average metrics across test sizes (0.1/0.2/0.3) ===")
display(summary)


## 6️⃣ Confusion Matrix & ROC Curve cho cấu hình tốt nhất

In [ ]:
def plot_confusion_and_roc(artifacts, dataset_name: str):
    # Chọn cấu hình tốt nhất theo F1 (ưu tiên test_size=0.2 nếu hòa)
    sub = results[results["dataset"]==dataset_name].copy()
    sub["ts_priority"] = (sub["test_size"]==0.2).astype(int)
    best_row = sub.sort_values(["f1","ts_priority","roc_auc"], ascending=[False, False, False]).iloc[0]

    ts = float(best_row["test_size"])
    model_name = best_row["model"]
    key = (dataset_name, ts, model_name)

    model, X_test, y_test, y_pred, y_prob = artifacts[key]

    print(f"Best config for {dataset_name}: model={model_name}, test_size={ts}")
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, digits=4))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix — {dataset_name} — {model_name}")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.show()

    # ROC curve
    if y_prob is not None:
        RocCurveDisplay.from_predictions(y_test, y_prob)
        plt.title(f"ROC Curve — {dataset_name} — {model_name}")
        plt.show()
    else:
        print("Model không có y_prob để vẽ ROC.")

plot_confusion_and_roc(artifacts_all, "combined")


### ✅ [CELL THÊM - ChatGPT] ROC curve cho **từng mô hình** (so sánh trực quan)
Cell này bổ sung phần **ROC curve cho từng mô hình** theo đúng mục đánh giá.

In [ ]:
# ✅ ROC curve cho từng mô hình (vẽ tất cả model trên cùng một biểu đồ)
from sklearn.metrics import roc_curve, auc

def plot_roc_curves_all_models(artifacts, dataset_name="all", test_size=0.2, title_suffix=""):
    # Lấy danh sách model có sẵn trong artifacts
    keys = [k for k in artifacts.keys() if k[0] == dataset_name and k[1] == test_size]
    if len(keys) == 0:
        print(f"Không tìm thấy artifacts cho dataset='{dataset_name}', test_size={test_size}.")
        return

    plt.figure(figsize=(6,5))
    plotted = 0

    for (_, _, model_name) in sorted(keys, key=lambda x: x[2]):
        model, X_test, y_test, y_pred, y_prob = artifacts[(dataset_name, test_size, model_name)]
        if y_prob is None:
            # Nếu không có proba/score thì bỏ qua ROC
            continue
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{model_name} (AUC={roc_auc:.3f})")
        plotted += 1

    # đường baseline
    plt.plot([0, 1], [0, 1], linestyle="--")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves — dataset={dataset_name}, test_size={test_size} {title_suffix}".strip())
    plt.legend()
    plt.tight_layout()
    plt.show()

    if plotted == 0:
        print("Không có model nào cung cấp xác suất/score để vẽ ROC.")

# Ưu tiên test_size=0.2 nếu có; nếu không có thì dùng test_size phổ biến nhất
preferred_ts = 0.2
available_ts = sorted(results["test_size"].unique().tolist()) if "results" in globals() else [preferred_ts]
ts_use = preferred_ts if preferred_ts in available_ts else available_ts[0]

# Vẽ ROC cho dataset 'all'
plot_roc_curves_all_models(artifacts_all if "artifacts_all" in globals() else artifacts,
                           dataset_name="all", test_size=ts_use)

# Nếu bạn có chạy so sánh đỏ/trắng thì vẽ thêm:
if "artifacts_red" in globals() and len(artifacts_red) > 0:
    plot_roc_curves_all_models(artifacts_red, dataset_name="red", test_size=ts_use)
if "artifacts_white" in globals() and len(artifacts_white) > 0:
    plot_roc_curves_all_models(artifacts_white, dataset_name="white", test_size=ts_use)


## 7️⃣ Feature importance (LightGBM & XGBoost)

In [ ]:
def plot_feature_importance_for_best(artifacts, dataset_name: str, model_name: str):
    # Lấy best row theo dataset + model với F1 cao nhất
    sub = results[(results["dataset"]==dataset_name) & (results["model"]==model_name)].copy()
    best_row = sub.sort_values(["f1","roc_auc"], ascending=[False, False]).iloc[0]
    ts = float(best_row["test_size"])
    model, X_test, y_test, y_pred, y_prob = artifacts[(dataset_name, ts, model_name)]

    # Model có thể là sklearn estimator hoặc Pipeline
    est = model
    if isinstance(model, Pipeline):
        est = model.named_steps.get("clf", model)

    if hasattr(est, "feature_importances_"):
        imp = pd.Series(est.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
        plt.figure(figsize=(8,4))
        imp.head(15).plot(kind="bar")
        plt.title(f"Feature importance — {dataset_name} — {model_name}")
        plt.ylabel("Importance")
        plt.tight_layout()
        plt.show()
        display(imp.head(15))
    else:
        print(f"{model_name} không hỗ trợ feature_importances_")

# combined: LightGBM & XGBoost
plot_feature_importance_for_best(artifacts_all, "combined", "LightGBM")
plot_feature_importance_for_best(artifacts_all, "combined", "XGBoost")


## 8️⃣ (Optional) Regression — dự đoán điểm quality

In [ ]:
# Regression là phần mở rộng (optional). Bạn có thể bỏ qua nếu không cần.

from sklearn.metrics import mean_squared_error, mean_absolute_error

def run_regression(df: pd.DataFrame, dataset_name: str, test_size=0.2):
    df_work = df.copy()
    X = df_work[FEATURE_COLS]
    y = df_work["quality"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_STATE
    )

    n_estimators = 400 if QUICK_RUN else 1200

    models = {
        "LGBMRegressor": LGBMRegressor(n_estimators=n_estimators, learning_rate=0.03, random_state=RANDOM_STATE, n_jobs=-1),
        "XGBRegressor": XGBRegressor(n_estimators=n_estimators, learning_rate=0.03, max_depth=4,
                                     subsample=0.8, colsample_bytree=0.8,
                                     random_state=RANDOM_STATE, tree_method="hist", n_jobs=-1),
    }

    rows = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        mae = mean_absolute_error(y_test, pred)
        rows.append({"dataset": dataset_name, "model": name, "RMSE": rmse, "MAE": mae})

    return pd.DataFrame(rows).sort_values("RMSE")

if RUN_COMPARE_RED_WHITE:
    reg_results = pd.concat([
        run_regression(df_all, "combined", test_size=0.2),
        run_regression(df_red, "red", test_size=0.2),
        run_regression(df_white, "white", test_size=0.2),
    ], ignore_index=True)
else:
    reg_results = run_regression(df_all, "combined", test_size=0.2)

display(reg_results)
